# LAB_ML01 — Solución guiada
## Pipeline end-to-end con Logistic Regression

Recorrido: Spark → pandas → train/test → scaler → Logistic Regression → métricas → MLflow → scoring.

## 1. Cargar el dataset Gold

La lectura se realiza con Spark para mantener el punto de entrada dentro del lakehouse.

In [ ]:
from pyspark.sql import functions as F

TRAIN_PATH = "/Volumes/training/shared/source/ml/customer_churn_gold.csv"
SCORING_PATH = "/Volumes/training/shared/source/ml/customer_churn_scoring.csv"

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(TRAIN_PATH)
)

display(df.limit(10))
df.printSchema()
print("Filas:", df.count())

display(
    df.groupBy("churn")
      .count()
      .orderBy("churn")
)

## 2. Identificar las 11 features numéricas

Excluimos el identificador y la etiqueta. El notebook comprueba que el dataset preparado contiene exactamente 11 variables numéricas.

In [ ]:
numeric_types = {"byte", "short", "int", "bigint", "long", "float", "double", "decimal"}

feature_cols = [
    field.name
    for field in df.schema.fields
    if field.name not in {"customer_id", "churn"}
    and (
        field.dataType.simpleString() in numeric_types
        or field.dataType.simpleString().startswith("decimal")
    )
]

print("Features:", feature_cols)
print("Número de features:", len(feature_cols))

assert len(feature_cols) == 11, (
    f"Se esperaban 11 features numéricas y se encontraron {len(feature_cols)}"
)

## 3. Pasar el subconjunto de entrenamiento a pandas

Para este dataset pequeño utilizaremos scikit-learn en el driver.

In [ ]:
pdf = df.select("customer_id", *feature_cols, "churn").toPandas()

X = pdf[feature_cols].astype("float64")
y = pdf["churn"].astype("int64")

print(X.shape)
print(y.value_counts(normalize=True).sort_index())

## 4. Separar train y test

La estratificación conserva aproximadamente la proporción de churn en ambos conjuntos.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

## 5. Construir el pipeline

El scaler queda dentro del pipeline para garantizar que entrenamiento e inferencia aplican el mismo preprocesamiento.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

## 6. Evaluar

Accuracy no es suficiente por sí sola. Calculamos también precision, recall, F1 y ROC-AUC.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred, zero_division=0),
    "recall": recall_score(y_test, pred, zero_division=0),
    "f1": f1_score(y_test, pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, proba),
}

print(metrics)
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, pred))
print("\nClassification report:")
print(classification_report(y_test, pred, zero_division=0))

### Interpretación

- **Precision**: de los clientes marcados como churn, cuántos realmente lo eran.
- **Recall**: de los clientes que realmente abandonan, cuántos detectamos.
- **F1**: equilibrio entre precision y recall.
- **ROC-AUC**: capacidad del modelo para ordenar positivos por encima de negativos a distintos umbrales.

## 7. Registrar el experimento con MLflow

In [ ]:
import mlflow
import mlflow.sklearn

with mlflow.start_run(run_name="LAB_ML01_LogisticRegression") as run:
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("num_features", len(feature_cols))
    mlflow.log_param("train_rows", len(X_train))
    mlflow.log_param("test_rows", len(X_test))
    mlflow.log_param("features", ",".join(feature_cols))

    for name, value in metrics.items():
        mlflow.log_metric(name, float(value))

    mlflow.sklearn.log_model(model, "model")

    run_id = run.info.run_id

print("MLflow Run ID:", run_id)

## 8. Scoring de nuevos clientes

In [ ]:
scoring_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(SCORING_PATH)
)

display(scoring_df.limit(10))
print("Filas scoring:", scoring_df.count())

In [ ]:
scoring_pdf = scoring_df.select("customer_id", *feature_cols).toPandas()

X_scoring = scoring_pdf[feature_cols].astype("float64")

scoring_pdf["predicted_churn"] = model.predict(X_scoring).astype(int)
scoring_pdf["churn_probability"] = model.predict_proba(X_scoring)[:, 1]

scored_spark = spark.createDataFrame(
    scoring_pdf[["customer_id", "predicted_churn", "churn_probability"]]
)

display(
    scored_spark.orderBy(F.desc("churn_probability"))
)

## 9. Resultado

Hemos recorrido un pipeline ML completo: datos gobernados → entrenamiento → evaluación → tracking → inferencia. El siguiente laboratorio mantiene el mismo problema y sustituye el modelo lineal por una red neuronal.